
# Sistema Completo de Medición de Peces con YOLOv8 y OpenCV

Este notebook integra **todo el flujo del proyecto de detección y medición de peces** usando **YOLOv8**.
Incluye tres etapas principales:

**Entrenamiento** con validación cruzada (K-Fold).  
**Validación externa** del mejor modelo.  
**Medición automática** de la longitud física de los peces (en centímetros).


## 1. Entrenamiento con validación cruzada (K-Fold)

In [1]:
from ultralytics import YOLO
import os
import torch
import math
import cv2

print('Verificando entorno y dispositivo...')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo activo: {DEVICE.upper()}')

BASE_PATH   = os.getcwd()
IMAGES_PATH = os.path.join(BASE_PATH, 'images')
RUNS_PATH   = os.path.join(BASE_PATH, 'runs_kfold')
VAL_IMAGES  = os.path.join(IMAGES_PATH, 'val')
OUTPUT_DIR  = os.path.join(BASE_PATH, 'val_final')
MODEL_BASE  = 'yolov8s.pt'
EPOCHS      = 20
IMG_SIZE    = 640
BATCH_SIZE  = 8

os.makedirs(RUNS_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Directorios configurados correctamente.')

Verificando entorno y dispositivo...
Dispositivo activo: CUDA
Directorios configurados correctamente.


## 2. Validación externa del modelo

In [2]:
def run_kfold(base_path, model_base, epochs, img_size, batch_size):
    print('Entrenando con validación cruzada K-Fold (5 folds)')
    folds = [1, 2, 3, 4, 5]

    for val_fold in folds:
        print(f'FOLD {val_fold}: Validando con fold{val_fold}, entrenando con los demás')
        train_folds = [f'images/fold{i}' for i in folds if i != val_fold]
        val_fold_path = f'images/fold{val_fold}'

        # Crear YAML de configuración para cada fold
        data_yaml = os.path.join(base_path, f'data_fold{val_fold}.yaml')
        with open(data_yaml, 'w', encoding='utf-8') as f:
            f.write(f"# Dataset YOLOv8 - Fold {val_fold}\n")
            f.write(f"path: {os.path.abspath(base_path).replace(os.sep, '/')}\n")
            f.write("train:\n")
            for fold in train_folds:
                f.write(f"  - {fold}\n")
            f.write(f"val: {val_fold_path}\n")
            f.write("names:\n")
            f.write("  0: fish\n")

        # Entrenar modelo en este fold
        model = YOLO(model_base)
        results = model.train(
            data=data_yaml,
            epochs=epochs,
            imgsz=img_size,
            batch=batch_size,
            project=os.path.join(base_path, 'runs_kfold'),
            name=f'fold{val_fold}_train',
            exist_ok=True,
            augment=True
        )

        # Cargar modelo entrenado y predecir sobre el fold de validación
        trained_model = YOLO(os.path.join(results.save_dir, 'weights', 'best.pt'))
        output_dir = os.path.join(base_path, f'val{val_fold}')
        os.makedirs(output_dir, exist_ok=True)
        trained_model.predict(
            source=val_fold_path,
            conf=0.25,
            imgsz=img_size,
            save=True,
            project=output_dir,
            name='',
            exist_ok=True
        )

        print(f'Fold {val_fold} completado. Resultados guardados en {output_dir}')

# Ejecutar la función principal
run_kfold(BASE_PATH, MODEL_BASE, EPOCHS, IMG_SIZE, BATCH_SIZE)


Entrenando con validación cruzada K-Fold (5 folds)
FOLD 1: Validando con fold1, entrenando con los demás
New https://pypi.org/project/ultralytics/8.3.223 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.0  Python-3.12.0 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: task=detect, mode=train, model=yolov8s.pt, data=C:\Users\ignac\Desktop\salmon\data_fold1.yaml, epochs=20, time=None, patience=100, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=C:\Users\ignac\Desktop\salmon\runs_kfold, name=fold1_train, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=Fal

train: Scanning C:\Users\ignac\Desktop\salmon\labels\fold2... 1054 images, 118 backgrounds, 0 corrupt: 100%|██████████| 1054/1054 [00:02<00:00, 354.17it/s]


train: New cache created: C:\Users\ignac\Desktop\salmon\labels\fold2.cache


val: Scanning C:\Users\ignac\Desktop\salmon\labels\fold1... 263 images, 20 backgrounds, 0 corrupt: 100%|██████████| 263/263 [00:01<00:00, 188.94it/s]

val: New cache created: C:\Users\ignac\Desktop\salmon\labels\fold1.cache


...

      20/20      2.23G      1.068     0.5039      1.178          7        640: 100%|██████████| 132/132 [00:16<00:00,  8.11it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 17/17 [00:01<00:00,  9.21it/s]

                   all        265        310      0.958      0.974      0.989      0.705



20 epochs completed in 0.114 hours.
Optimizer stripped from C:\Users\ignac\Desktop\salmon\runs_kfold\fold5_train\weights\last.pt, 19.9MB
Optimizer stripped from C:\Users\ignac\Desktop\salmon\runs_kfold\fold5_train\weights\best.pt, 19.9MB

Validating C:\Users\ignac\Desktop\salmon\runs_kfold\fold5_train\weights\best.pt...
Ultralytics 8.3.0  Python-3.12.0 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
Model summary (fused): 186 layers, 9,828,051 parameters, 0 gradients, 23.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 17/17 [00:03<00:00,  4.89it/s]


                   all        265        310      0.915      0.976      0.983      0.696
Speed: 0.3ms preprocess, 8.1ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to C:\Users\ignac\Desktop\salmon\runs_kfold\fold5_train

image 1/265 C:\Users\ignac\Desktop\salmon\images\fold5\modificacion_002.jpeg: 480x640 (no detections), 21.0ms
image 2/265 C:\Users\ignac\Desktop\salmon\images\fold5\modificacion_005.jpeg: 480x640 1 fish, 18.0ms
image 3/265 C:\Users\ignac\Desktop\salmon\images\fold5\modificacion_007.jpeg: 480x640 1 fish, 17.0ms
image 4/265 C:\Users\ignac\Desktop\salmon\images\fold5\modificacion_014.jpeg: 480x640 2 fishs, 17.0ms
image 5/265 C:\Users\ignac\Desktop\salmon\images\fold5\modificacion_015.jpeg: 480x640 2 fishs, 17.0ms
image 6/265 C:\Users\ignac\Desktop\salmon\images\fold5\modificacion_024.jpeg: 480x640 (no detections), 18.0ms
image 7/265 C:\Users\ignac\Desktop\salmon\images\fold5\modificacion_031.jpeg: 480x640 1 fish, 18.0ms
image 8/265 C:\Users\ignac\Desk

## 3. Medición automática de peces con YOLO + OpenCV

In [3]:
def find_best_model(base_path):
    for root, _, files in os.walk(base_path):
        for file in files:
            if file == 'best.pt':
                return os.path.join(root, file)
    return None

def validate_final_external():
    print('Buscando el mejor modelo en los folds...')
    best_model = find_best_model(RUNS_PATH)
    if not best_model:
        print('No se encontró ningún modelo best.pt en runs_kfold.')
        return
    print(f'Modelo seleccionado: {best_model}')
    model = YOLO(best_model)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    results = model.predict(
        source=VAL_IMAGES,
        conf=0.25,
        imgsz=IMG_SIZE,
        device=DEVICE,
        save=True,
        project=OUTPUT_DIR,
        name='',
        exist_ok=True
    )
    print(f'Validación externa completada. Resultados guardados en: {OUTPUT_DIR}')

validate_final_external()

Buscando el mejor modelo en los folds...
Modelo seleccionado: C:\Users\ignac\Desktop\salmon\runs_kfold\fold1_train\weights\best.pt

image 1/526 C:\Users\ignac\Desktop\salmon\images\val\modificacion_004.jpeg: 480x640 1 fish, 55.0ms
image 2/526 C:\Users\ignac\Desktop\salmon\images\val\modificacion_009.jpeg: 480x640 2 fishs, 28.0ms
image 3/526 C:\Users\ignac\Desktop\salmon\images\val\modificacion_010.jpeg: 480x640 2 fishs, 54.0ms
image 4/526 C:\Users\ignac\Desktop\salmon\images\val\modificacion_020.jpeg: 480x640 2 fishs, 53.0ms
image 5/526 C:\Users\ignac\Desktop\salmon\images\val\modificacion_025.jpeg: 480x640 (no detections), 26.0ms
image 6/526 C:\Users\ignac\Desktop\salmon\images\val\modificacion_026.jpeg: 480x640 1 fish, 25.0ms
image 7/526 C:\Users\ignac\Desktop\salmon\images\val\modificacion_029.jpeg: 480x640 1 fish, 26.0ms
image 8/526 C:\Users\ignac\Desktop\salmon\images\val\modificacion_037.jpeg: 480x640 1 fish, 26.0ms
image 9/526 C:\Users\ignac\Desktop\salmon\images\val\modificacio

In [4]:

from ultralytics import YOLO
import cv2
import numpy as np
import os
import torch

MODEL_PATH = "runs_kfold/fold3_train/weights/best.pt"
VAL_IMAGES_DIR = "images/val"
OUTPUT_DIR = "val_medidos_auto"

BANDEJA_CM = 50.0
BANDEJA_PX = 1456.0
FACTOR_CM = BANDEJA_CM / BANDEJA_PX

os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE.upper()}")

def choose_measurement(width, height):
    if width > 1.5 * height:
        return "horizontal"
    elif height > 1.5 * width:
        return "vertical"
    else:
        return "diagonal"

def measure_bbox(x1, y1, x2, y2, mode):
    width = x2 - x1
    height = y2 - y1
    if mode == "horizontal":
        length_px = width
    elif mode == "vertical":
        length_px = height
    else:
        length_px = np.hypot(width, height)
    return length_px * FACTOR_CM

def draw_selected_line(img, x1, y1, x2, y2, mode, length_cm):
    xm = (x1 + x2) // 2
    ym = (y1 + y2) // 2
    if mode == "horizontal":
        cv2.line(img, (x1, ym), (x2, ym), (0, 255, 0), 2)
        color_text = (0, 255, 0)
    elif mode == "vertical":
        cv2.line(img, (xm, y1), (xm, y2), (0, 255, 255), 2)
        color_text = (0, 255, 255)
    else:
        cv2.line(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.line(img, (x1, y2), (x2, y1), (0, 0, 255), 2)
        color_text = (255, 255, 0)
    cv2.rectangle(img, (x1, y1), (x2, y2), (50, 255, 50), 2)
    cv2.putText(img, f"{length_cm:.1f} cm ({mode})",
                (x1, max(0, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX,
                0.6, color_text, 2)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"No se encontró el modelo en: {MODEL_PATH}")

model = YOLO(MODEL_PATH)
print("Modelo YOLO cargado correctamente.")

for img_name in os.listdir(VAL_IMAGES_DIR):
    if not img_name.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    img_path = os.path.join(VAL_IMAGES_DIR, img_name)
    img = cv2.imread(img_path)
    if img is None:
        print(f"No se pudo leer: {img_name}")
        continue

    results = model(img, verbose=False, device=DEVICE)
    detections = 0

    for r in results:
        if not hasattr(r, "boxes"):
            continue

        for box in r.boxes:
            detections += 1
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            width = x2 - x1
            height = y2 - y1
            mode = choose_measurement(width, height)
            length_cm = measure_bbox(x1, y1, x2, y2, mode)
            draw_selected_line(img, x1, y1, x2, y2, mode, length_cm)
            print(f"{img_name}: {length_cm:.2f} cm ({mode})")

    if detections == 0:
        print(f"{img_name}: sin detecciones")

    cv2.imwrite(os.path.join(OUTPUT_DIR, img_name), img)

print("Mediciones automáticas completadas.")


Dispositivo: CUDA
Modelo YOLO cargado correctamente.
modificacion_004.jpeg: 7.35 cm (vertical)
modificacion_009.jpeg: 11.57 cm (vertical)
modificacion_009.jpeg: 4.09 cm (diagonal)
modificacion_010.jpeg: 11.47 cm (vertical)
modificacion_010.jpeg: 5.84 cm (vertical)
modificacion_020.jpeg: 11.50 cm (vertical)
modificacion_020.jpeg: 10.28 cm (diagonal)
modificacion_025.jpeg: sin detecciones
modificacion_026.jpeg: 6.19 cm (diagonal)
modificacion_029.jpeg: 9.03 cm (vertical)
modificacion_037.jpeg: 13.77 cm (vertical)
modificacion_041.jpeg: 13.32 cm (vertical)
modificacion_043.jpeg: 12.95 cm (vertical)
modificacion_046.jpeg: 12.77 cm (vertical)
modificacion_053.jpeg: 13.63 cm (vertical)
modificacion_053.jpeg: 8.93 cm (diagonal)
modificacion_071.jpeg: 11.54 cm (horizontal)
modificacion_072.jpeg: 11.26 cm (horizontal)
modificacion_076.jpeg: sin detecciones
modificacion_078.jpeg: 13.29 cm (horizontal)
modificacion_082.jpeg: 14.19 cm (diagonal)
modificacion_095.jpeg: 14.05 cm (diagonal)
modificac